# TCGA-BRCA Multimodal Stratification — 01 · Download

Unsupervised stratification of breast-cancer patients from **two genome-wide molecular modalities** plus clinical/survival data:

- **expression** — gene expression (RNA-seq, IlluminaHiSeq, `log2(norm_count + 1)`)
- **cnv** — copy number, gene-level GISTIC2 (continuous)
- **clinical** — phenotype matrix (survival fields + PAM50 subtype)

This notebook only **downloads** the raw data from the UCSC Xena public hubs and saves it to `data/raw/`.
Transposing, aligning samples, feature selection and scaling all happen in notebook `02`.

> Runs on your machine: it fetches from Xena over the network. The raw files are **not** committed to git (see `.gitignore`).

In [3]:
import requests
from pathlib import Path
import pandas as pd

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

## The three datasets

UCSC Xena TCGA hub, cohort **TCGA Breast Cancer (BRCA)** (`TCGA.BRCA.sampleMap`).

We take the **continuous** GISTIC2 copy number (`all_data_by_genes`), not the thresholded `-2..+2` version: continuous values standardize and reduce with PCA far better than four discrete levels.

> If any URL ever 404s, open [xenabrowser.net/datapages](https://xenabrowser.net/datapages/), pick the BRCA cohort, find the dataset, and paste its download URL here.

In [4]:
BASE = "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap"

XENA_FILES = {
    "expression": f"{BASE}/HiSeqV2.gz",
    "cnv":        f"{BASE}/Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz",
    "clinical":   f"{BASE}/BRCA_clinicalMatrix",   # plain text, not gzipped
}

In [12]:
XENA_FILES

{'expression': 'https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/HiSeqV2.gz',
 'cnv': 'https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz',
 'clinical': 'https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/BRCA_clinicalMatrix'}

## Download helper

`stream=True` plus 1 MB chunks means a big matrix (~20k genes × ~1200 samples) never sits fully in memory while downloading. If a file is already on disk, we skip it — no re-downloading on every run.

In [5]:
def download_file(url: str, dest: Path) -> None:
    """Stream a file from Xena to disk. Skips if we already have it."""
    if dest.exists():
        print(f"  already have {dest.name}, skipping")
        return
    print(f"  downloading {dest.name} ...")
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB at a time
                f.write(chunk)
    print(f"    saved -> {dest}")

## Run the downloads

In [6]:
paths = {}
for name, url in XENA_FILES.items():
    ext = ".gz" if url.endswith(".gz") else ".tsv"
    dest = RAW_DIR / f"{name}{ext}"
    download_file(url, dest)
    paths[name] = dest

paths

  already have expression.gz, skipping
  already have cnv.gz, skipping
  already have clinical.tsv, skipping


{'expression': PosixPath('data/raw/expression.gz'),
 'cnv': PosixPath('data/raw/cnv.gz'),
 'clinical': PosixPath('data/raw/clinical.tsv')}

## Sanity check: orientation + size

On Xena the two genomic matrices are stored as **genes (rows) × samples (columns)** — the opposite of what we need (one patient per row). We keep them raw for now and transpose in notebook `02`. Here we just confirm they read correctly.

In [13]:
def peek(name: str, path: Path) -> None:
    """Read the first few rows to confirm orientation and width."""
    compression = "gzip" if path.suffix == ".gz" else None
    head = pd.read_csv(path, sep="\t", index_col=0, compression=compression, nrows=5)
    print(f"  {name:11s} row label = '{head.index.name}', {head.shape[1]} columns")

for name, path in paths.items():
    try:
        peek(name, path)
    except Exception as e:
        print(f"  could not read {name}: {e}")

  expression  row label = 'sample', 1218 columns
  cnv         row label = 'Gene Symbol', 1080 columns
  clinical    row label = 'sampleID', 193 columns


## List the clinical columns

The survival and PAM50 field names vary between Xena versions, so instead of guessing them we list them here — and use this to pick the exact columns in notebook `02`.

In [19]:
clin = pd.read_csv(paths["clinical"], sep="\t", index_col=0, nrows=1)
for c in sorted(clin.columns):
    print(c)

AJCC_Stage_nature2012
Age_at_Initial_Pathologic_Diagnosis_nature2012
CN_Clusters_nature2012
Converted_Stage_nature2012
Days_to_Date_of_Last_Contact_nature2012
Days_to_date_of_Death_nature2012
ER_Status_nature2012
Gender_nature2012
HER2_Final_Status_nature2012
Integrated_Clusters_no_exp__nature2012
Integrated_Clusters_unsup_exp__nature2012
Integrated_Clusters_with_PAM50__nature2012
Metastasis_Coded_nature2012
Metastasis_nature2012
Node_Coded_nature2012
Node_nature2012
OS_Time_nature2012
OS_event_nature2012
PAM50Call_RNAseq
PAM50_mRNA_nature2012
PR_Status_nature2012
RPPA_Clusters_nature2012
SigClust_Intrinsic_mRNA_nature2012
SigClust_Unsupervised_mRNA_nature2012
Survival_Data_Form_nature2012
Tumor_T1_Coded_nature2012
Tumor_nature2012
Vital_Status_nature2012
_GENOMIC_ID_TCGA_BRCA_G4502A_07_3
_GENOMIC_ID_TCGA_BRCA_PDMRNAseq
_GENOMIC_ID_TCGA_BRCA_PDMRNAseqCNV
_GENOMIC_ID_TCGA_BRCA_PDMarray
_GENOMIC_ID_TCGA_BRCA_PDMarrayCNV
_GENOMIC_ID_TCGA_BRCA_RPPA
_GENOMIC_ID_TCGA_BRCA_RPPA_RBN
_GENOMIC_I

## Next

Once this has run, note down:
1. the three shapes from the sanity check, and
2. the clinical column names (especially anything that looks like survival — `OS`, `days_to_*`, `vital_status` — and PAM50).

We pick the exact survival + subtype columns from that list and build **`02 · preprocess`** on it.